# Building a Gemini Client — Step by Step

Same journey as the OpenAI notebook, but for **Google Gemini**: start with one call that gets an answer, then wrap it in a reusable async class with non-streaming, streaming, retries, and error handling.

Everything is normalised to plain text so the rest of the project can use Gemini and OpenAI interchangeably.

## 0. Setup

In [5]:
%pip install google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
from dotenv import load_dotenv

# Put GEMINI_API_KEY (and optionally GEMINI_MODEL) in a .env file next to this notebook.
load_dotenv()

# Option B: set it directly here (remove this after testing!)
# os.environ['GEMINI_API_KEY'] = 'AIza...'
# os.environ['GEMINI_MODEL'] = 'gemini-2.5-flash'

print('api key :', bool(os.getenv('GEMINI_API_KEY')))
print('model   :', os.getenv('GEMINI_MODEL', 'gemini-2.5-flash'))

api key : True
model   : gemini-3.6-flash


## 1. Just the LLM call — get the answer

No class, no wrapper. Create the client, send one prompt, print the answer.

Gemini's SDK is async-capable via `client.aio`, so we `await` the call — exactly the same shape as the OpenAI client.

In [7]:
import asyncio
from google import genai


async def main():
    client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

    response = await client.aio.models.generate_content(
        model=os.getenv('GEMINI_MODEL', 'gemini-2.5-flash'),
        contents='tell me about the moon in one sentence',
    )

    print('Answer:', response.text)
    client.close()


await main()

Answer: The Moon is Earth’s only natural satellite, a crater-pocked rocky world that stabilizes our planet's climate, drives ocean tides, and stands as the only celestial body beyond Earth where humans have walked.


## 2. Look at the raw response

Before wrapping things in a class, let's see what Gemini actually returns: `model_version`, `response_id`, `finish_reason`, `role`, and `usage_metadata` (token counts).

In [8]:
async def main():
    client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

    response = await client.aio.models.generate_content(
        model=os.getenv('GEMINI_MODEL', 'gemini-2.5-flash'),
        contents='tell me about the moon in one sentence',
    )

    candidate = response.candidates[0]

    print('model_version:', response.model_version)
    print('response_id  :', response.response_id)
    print('create_time  :', response.create_time)
    print('finish_reason:', candidate.finish_reason)
    print('role         :', candidate.content.role)
    if response.usage_metadata:
        u = response.usage_metadata
        print('usage        : prompt={} candidates={} total={}'.format(
            u.prompt_token_count, u.candidates_token_count, u.total_token_count,
        ))

    print('\nAnswer:', response.text)
    client.close()


await main()

model_version: gemini-3.6-flash
response_id  : thquaqTtCYzajuMP48OmmQg
create_time  : None
finish_reason: FinishReason.STOP
role         : model
usage        : prompt=9 candidates=34 total=429

Answer: The Moon is Earth's only natural satellite, a cratered and airless world whose gravitational pull drives ocean tides while its surface reflects sunlight to illuminate our night sky.


### 2.1 Full response object — every parameter

The SDK returns pydantic objects, so `response.model_dump()` gives us everything the API sent back: `candidates` (with `content.parts`), `finish_reason`, `usage_metadata`, and more.

In [9]:
import json


async def main():
    client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

    response = await client.aio.models.generate_content(
        model=os.getenv('GEMINI_MODEL', 'gemini-2.5-flash'),
        contents='tell me about the moon in one sentence',
    )

    # dump EVERYTHING the API returned
    print(json.dumps(response.model_dump(), indent=2, default=str))
    client.close()


await main()

{
  "sdk_http_response": {
    "headers": {
      "x-gemini-service-tier": "standard",
      "content-type": "application/json; charset=UTF-8",
      "vary": "Origin, X-Origin, Referer",
      "content-encoding": "gzip",
      "date": "Sat, 19 Sep 2026 05:16:49 GMT",
      "server": "scaffolding on HTTPServer2",
      "x-xss-protection": "0",
      "x-frame-options": "SAMEORIGIN",
      "x-content-type-options": "nosniff",
      "server-timing": "gfet4t7; dur=5507",
      "alt-svc": "h3=\":443\"; ma=2592000,h3-29=\":443\"; ma=2592000",
      "transfer-encoding": "chunked"
    },
    "body": null
  },
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "media_resolution": null,
            "code_execution_result": null,
            "executable_code": null,
            "file_data": null,
            "function_call": null,
            "function_response": null,
            "inline_data": null,
            "text": "The Moon is Earth's only natural satellit

### 2.2 Full stream chunk — every parameter

The same dump, but for a **streaming** response. Each chunk has a `candidates` list; the text arrives in `parts[].text`, and `finish_reason` is only set on the final chunk.

In [10]:
import json


async def main():
    client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

    stream = await client.aio.models.generate_content_stream(
        model=os.getenv('GEMINI_MODEL', 'gemini-2.5-flash'),
        contents='tell me about the moon in one sentence',
    )

    async for chunk in stream:
        print(json.dumps(chunk.model_dump(), indent=2, default=str))
        print('-' * 40)

    client.close()


await main()

{
  "sdk_http_response": {
    "headers": {
      "content-type": "text/event-stream",
      "content-disposition": "attachment",
      "vary": "Origin, X-Origin, Referer",
      "transfer-encoding": "chunked",
      "date": "Sat, 19 Sep 2026 05:16:59 GMT",
      "server": "scaffolding on HTTPServer2",
      "x-xss-protection": "0",
      "x-frame-options": "SAMEORIGIN",
      "x-content-type-options": "nosniff",
      "server-timing": "gfet4t7; dur=3275",
      "alt-svc": "h3=\":443\"; ma=2592000,h3-29=\":443\"; ma=2592000"
    },
    "body": null
  },
  "candidates": [
    {
      "content": {
        "parts": [
          {
            "media_resolution": null,
            "code_execution_result": null,
            "executable_code": null,
            "file_data": null,
            "function_call": null,
            "function_response": null,
            "inline_data": null,
            "text": "The Moon is Earth's only natural satellite, a cratered, airless",
            "thought": 

## 3. Wrap it in a class (skeleton)

Now we move the call into a reusable `GeminiLLMClient` with a **lazy client** — the `genai.Client` is created only when first needed.

In [12]:
class GeminiLLMClient:
    """Step 3: class skeleton with a lazy Gemini client."""

    def __init__(self, api_key=None, model=None, max_retries=3):
        self._api_key = api_key or os.getenv('GEMINI_API_KEY')
        self._model = model or os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')
        self._max_retries = max_retries
        self._client = None

    def get_client(self):
        """Lazily build the underlying genai.Client."""
        if self._client is None:
            self._client = genai.Client(api_key=self._api_key)
        return self._client

    async def close(self):
        """Release the underlying resources (Gemini's close() is synchronous)."""
        if self._client is not None:
            self._client.close()
            self._client = None

In [13]:
client = GeminiLLMClient()

print('model    :', client._model)
print('has key  :', bool(client._api_key))
print('client   :', type(client.get_client()).__name__)

model    : gemini-3.6-flash
has key  : True
client   : Client


## 4. Non-streaming response

`_non_stream_response()` sends one request and gets the whole answer back, printing the raw response parameters.

We also add `_build_request()`, which converts OpenAI-style dict messages into Gemini `contents` + config: `system` goes to `system_instruction`, `assistant` becomes the `model` role.

In [14]:
from google.genai import types


class GeminiLLMClient:
    """Step 4: adds message conversion + the non-streaming completion."""

    def __init__(self, api_key=None, model=None, max_retries=3):
        self._api_key = api_key or os.getenv('GEMINI_API_KEY')
        self._model = model or os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')
        self._max_retries = max_retries
        self._client = None

    def get_client(self):
        if self._client is None:
            self._client = genai.Client(api_key=self._api_key)
        return self._client

    async def close(self):
        if self._client is not None:
            self._client.close()
            self._client = None

    def _build_request(self, messages):
        """OpenAI-style dicts -> Gemini contents + config."""
        contents = []
        system_parts = []
        for message in messages:
            role = message.get('role', 'user')
            content = message.get('content') or ''
            if role == 'system':
                system_parts.append(content)
                continue
            contents.append(types.Content(
                role='model' if role == 'assistant' else 'user',
                parts=[types.Part(text=content)],
            ))
        config = types.GenerateContentConfig(
            system_instruction='\n\n'.join(system_parts) if system_parts else None,
        )
        return contents, config

    async def chat_completion(self, messages, stream=False):
        """Step 4: non-streaming only."""
        client = self.get_client()
        contents, config = self._build_request(messages)

        response = await client.aio.models.generate_content(
            model=self._model, contents=contents, config=config,
        )
        candidate = response.candidates[0]

        # --- show the raw response with its inside parameters ---
        print('\n--- raw response ---')
        print('  model_version:', response.model_version)
        print('  response_id  :', response.response_id)
        print('  finish_reason:', candidate.finish_reason)
        print('  role         :', candidate.content.role)
        if response.usage_metadata:
            u = response.usage_metadata
            print('  usage        : prompt={} candidates={} total={}'.format(
                u.prompt_token_count, u.candidates_token_count, u.total_token_count,
            ))
        # --------------------------------------------------------

        return response.text

In [15]:
client = GeminiLLMClient()
messages = [
    {'role': 'system', 'content': 'You are concise.'},
    {'role': 'user', 'content': 'tell me about the moon in one sentence'},
]


async def run_non_stream():
    text = await client.chat_completion(messages, stream=False)
    print('\n[assistant]', text)
    await client.close()


await run_non_stream()


--- raw response ---
  model_version: gemini-3.6-flash
  response_id  : 2xquauKfDe_6juMP4YyZmQw
  finish_reason: FinishReason.STOP
  role         : model
  usage        : prompt=14 candidates=32 total=487

[assistant] The Moon is Earth's only natural satellite, a cratered, airless world whose gravitational pull drives our ocean tides and stabilizes the planet's climate.


## 5. Streaming response

`_stream_response()` is an **async generator**: each chunk of text is yielded as it arrives. The raw chunk parameters are printed first — note `finish_reason` is `None` on intermediate chunks and only set on the final one.

This step also adds the full `chat_completion()` dispatcher plus retry logic (exponential backoff on rate limits / server / connection errors). We also disable Gemini's *automatic function calling* — this client is text-only.

In [16]:
import asyncio
from google import genai
from google.genai import errors, types


class GeminiLLMClient:
    """Step 5: full client - streaming + non-streaming + retry/backoff."""

    def __init__(self, api_key=None, model=None, max_retries=3):
        self._api_key = api_key or os.getenv('GEMINI_API_KEY')
        self._model = model or os.getenv('GEMINI_MODEL', 'gemini-2.5-flash')
        self._max_retries = max_retries
        self._client = None

    def get_client(self):
        if self._client is None:
            self._client = genai.Client(api_key=self._api_key)
        return self._client

    async def close(self):
        if self._client is not None:
            self._client.close()
            self._client = None

    def _build_request(self, messages):
        contents = []
        system_parts = []
        for message in messages:
            role = message.get('role', 'user')
            content = message.get('content') or ''
            if role == 'system':
                system_parts.append(content)
                continue
            contents.append(types.Content(
                role='model' if role == 'assistant' else 'user',
                parts=[types.Part(text=content)],
            ))
        config = types.GenerateContentConfig(
            system_instruction='\n\n'.join(system_parts) if system_parts else None,
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        )
        return contents, config

    async def chat_completion(self, messages, stream=True):
        """Yield plain text chunks, retrying with exponential backoff."""
        client = self.get_client()
        contents, config = self._build_request(messages)

        for attempt in range(self._max_retries + 1):
            try:
                if stream:
                    async for chunk in self._stream_response(client, contents, config):
                        yield chunk
                else:
                    chunk = await self._non_stream_response(client, contents, config)
                    if chunk:
                        yield chunk
                return

            except errors.ClientError as e:
                if getattr(e, 'code', None) == 429 and attempt < self._max_retries:
                    await asyncio.sleep(2 ** attempt)  # 1s, 2s, 4s...
                    continue
                yield f'[error] client error: {e}'
                return

            except errors.ServerError as e:
                if attempt < self._max_retries:
                    await asyncio.sleep(2 ** attempt)
                    continue
                yield f'[error] server error: {e}'
                return

            except Exception as e:
                if attempt < self._max_retries:
                    await asyncio.sleep(2 ** attempt)
                    continue
                yield f'[error] connection error: {e}'
                return

    @staticmethod
    def _text_from(candidate):
        """Join the text of a candidate's non-thought parts."""
        if candidate.content is None or not candidate.content.parts:
            return ''
        return ''.join(
            part.text for part in candidate.content.parts
            if getattr(part, 'text', None) and not getattr(part, 'thought', False)
        )

    async def _stream_response(self, client, contents, config):
        """Stream text chunks and print raw chunk parameters."""
        stream = await client.aio.models.generate_content_stream(
            model=self._model, contents=contents, config=config,
        )

        async for chunk in stream:
            if not chunk.candidates:
                continue
            candidate = chunk.candidates[0]

            # --- raw stream chunk parameters ---
            print('\n--- raw stream chunk ---')
            print('  model_version:', chunk.model_version)
            print('  finish_reason:', candidate.finish_reason)
            if chunk.usage_metadata:
                u = chunk.usage_metadata
                print('  usage        : prompt={} candidates={} total={}'.format(
                    u.prompt_token_count, u.candidates_token_count, u.total_token_count,
                ))
            # -----------------------------------

            text = self._text_from(candidate)
            if text:
                yield text

    async def _non_stream_response(self, client, contents, config):
        """Get the full response at once and print raw response parameters."""
        response = await client.aio.models.generate_content(
            model=self._model, contents=contents, config=config,
        )
        candidate = response.candidates[0]

        print('\n--- raw response ---')
        print('  model_version:', response.model_version)
        print('  response_id  :', response.response_id)
        print('  finish_reason:', candidate.finish_reason)
        print('  role         :', candidate.content.role)
        if response.usage_metadata:
            u = response.usage_metadata
            print('  usage        : prompt={} candidates={} total={}'.format(
                u.prompt_token_count, u.candidates_token_count, u.total_token_count,
            ))

        return self._text_from(candidate)

In [17]:
client = GeminiLLMClient()
messages = [{'role': 'user', 'content': 'tell me about the moon in one sentence'}]


async def run_stream():
    async for chunk in client.chat_completion(messages, stream=True):
        print(chunk, end='', flush=True)
    print()
    await client.close()


await run_stream()


--- raw stream chunk ---
  model_version: gemini-3.6-flash
  finish_reason: None
  usage        : prompt=9 candidates=1 total=448
The
--- raw stream chunk ---
  model_version: gemini-3.6-flash
  finish_reason: None
  usage        : prompt=9 candidates=29 total=476
 Moon is Earth's only natural satellite, a cratered, rocky world that drives our ocean tides, illuminates the night sky, and remains
--- raw stream chunk ---
  model_version: gemini-3.6-flash
  finish_reason: None
  usage        : prompt=9 candidates=40 total=487
 the only celestial body beyond Earth ever visited by humans.
--- raw stream chunk ---
  model_version: gemini-3.6-flash
  finish_reason: FinishReason.STOP
  usage        : prompt=9 candidates=40 total=487



## 6. Full demo

A quick interactive demo combining everything. Try both `stream=True` and `stream=False`.

In [18]:
client = GeminiLLMClient()


async def ask(question, stream):
    messages = [{'role': 'user', 'content': question}]
    print(f'\n[user] {question}')
    print('[assistant] ', end='', flush=True)
    async for chunk in client.chat_completion(messages, stream=stream):
        print(chunk, end='', flush=True)
    print()


await ask('what is the capital of France?', stream=False)
await ask('and why is it famous?', stream=True)

await client.close()


[user] what is the capital of France?
[assistant] 
--- raw response ---
  model_version: gemini-3.6-flash
  response_id  : -Rquap71FLfjjuMP5eXM4Ak
  finish_reason: FinishReason.STOP
  role         : model
  usage        : prompt=8 candidates=8 total=111
The capital of France is **Paris**.

[user] and why is it famous?
[assistant] 
--- raw stream chunk ---
  model_version: gemini-3.6-flash
  finish_reason: None
  usage        : prompt=7 candidates=10 total=515
It looks like your question is missing the context!
--- raw stream chunk ---
  model_version: gemini-3.6-flash
  finish_reason: None
  usage        : prompt=7 candidates=37 total=542
 Could you please specify what **"it"** refers to? 

Tell me the name of the **place, person, artwork
--- raw stream chunk ---
  model_version: gemini-3.6-flash
  finish_reason: None
  usage        : prompt=7 candidates=63 total=568
, event, or object** you're asking about, and I'd be happy to explain why it's famous!
--- raw stream chunk ---
  model

## 7. Using it with the project's `StreamEvent` client

The notebook above built a **plain-text** client for learning. The project's `client/gemini.py` wraps the same logic but emits the common `StreamEvent` objects (from `llm/response.py`), so the `Agent` can drive Gemini and OpenAI the exact same way — no provider-specific code in the agent.

Run this cell from the project root (or make sure `src/` is on `sys.path`).

In [19]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

from agent.agent import Agent
from agent.event import AgentEventType
from client import ClientFactory

client = ClientFactory.create_client('gemini')
agent = Agent(client, system_prompt='You are terse.')

async with client:
    async for event in agent.run('Say hi in two words.'):
        if event.type is AgentEventType.TEXT_DELTA:
            print(event.data['content'], end='', flush=True)
        elif event.type is AgentEventType.AGENT_END:
            print('\nfinish:', event.data.get('finish_reason'))

Hello there.
finish: stop


## Recap

- **Step 1** — bare Gemini call that gets the answer
- **Step 2** — inspect the raw response (`model_version`, `finish_reason`, `usage_metadata`) and the full stream chunk
- **Step 3** — class skeleton with a lazy `genai.Client`
- **Step 4** — message conversion + non-streaming response
- **Step 5** — streaming response + retry/backoff
- **Step 6** — full demo
- **Step 7** — the project's `StreamEvent` client + `Agent`

Key Gemini differences from OpenAI: use `client.aio` for async, text lives in `candidates[0].content.parts[].text`, there is no `system` role in `contents` (use `config.system_instruction`), and the assistant role is `model`.

Set `GEMINI_API_KEY` (and optionally `GEMINI_MODEL`) in `.env` to run these cells.